# 🧊 实验 3 —— Frozen Lake 的最优策略

在本次实验中，我们将继续基于简化版 **FrozenLake** 环境，探索马尔可夫决策过程（MDP）。在实验 2 的基础上，本实验将重点练习以下三个核心内容：

1. **迭代策略评估（Iterative Policy Evaluation）**  
我们通过反复应用贝尔曼期望更新（Bellman expectation update），对给定策略的状态价值进行迭代计算，直到收敛。同时，将其与实验 2 中的闭式解进行对比。

2. **蒙特卡洛模拟（Monte Carlo Simulation）**  
在相同策略下，通过在 FrozenLake 环境中运行大量回合（episodes），从经验角度估计各状态的价值，并将这些估计结果与解析解进行对比。

3. **寻找最优策略（值迭代，Value Iteration）**  
我们将应用值迭代方法，计算最优状态价值函数，并据此提取能够最大化长期回报的最优策略。

In [ ]:
import gymnasium as gym
import numpy as np
np.set_printoptions(precision=2, suppress=True)
import random
from gymnasium.envs.toy_text.frozen_lake import generate_random_map

In [ ]:
# Define a smaller 3x3 map
DESC_3x3 = [
    "SFF",
    "FHF",
    "FFG",
]
env = gym.make("FrozenLake-v1",desc=DESC_3x3,is_slippery=True, render_mode="ansi")
obs, info = env.reset(seed=42)
#print(env.render())

In [ ]:
# 通用的 FrozenLake 风格地图转移构建器（适用于任意矩形地图）
# 与 Gymnasium 的动作顺序兼容：左=0，下=1，右=2，上=3

from typing import List, Tuple
import numpy as np

LEFT, DOWN, RIGHT, UP = 0, 1, 2, 3
ACTIONS = [LEFT, DOWN, RIGHT, UP]
DIRS = {
    LEFT:  (0, -1),
    DOWN:  (1, 0),
    RIGHT: (0, 1),
    UP:    (-1, 0),
}

def _grid_to_idx(r: int, c: int, ncols: int) -> int:
    return r * ncols + c

def _idx_to_grid(s: int, ncols: int) -> Tuple[int, int]:
    return divmod(s, ncols)

def _clip_move(r: int, c: int, dr: int, dc: int, nrows: int, ncols: int) -> Tuple[int, int]:
    rr, cc = r + dr, c + dc
    rr = min(max(rr, 0), nrows - 1)
    cc = min(max(cc, 0), ncols - 1)
    return rr, cc

def build_frozenlake_transitions(desc: List[str], is_slippery: bool = True):
    """
    Build transition probabilities for a FrozenLake-like grid.

    Args:
        desc: list of strings (rows), made of {'S','F','H','G'}.
        is_slippery: if True -> stochastic: {left, forward, right} each with prob 1/3.
                     if False -> deterministic in the intended direction.

    Returns:
        P: np.ndarray (S, A, S), transition probabilities
        R: np.ndarray (S, A, S), rewards (1 on entering 'G', else 0)
        absorbing: np.ndarray (S,), True for 'H' or 'G' (absorbing/self-loop)
        shape_2d: (nrows, ncols)
        flatten_map: np.ndarray (S,), identity map (r,c)->s ordering
    """
    nrows = len(desc)
    ncols = len(desc[0])
    S = nrows * ncols
    A = 4

    grid = np.array([list(row) for row in desc])
    is_hole = (grid == 'H')
    is_goal = (grid == 'G')
    absorbing = (is_hole | is_goal).reshape(-1)

    P = np.zeros((S, A, S), dtype=float)
    R = np.zeros((S, A, S), dtype=float)

    def step_from_state(s: int, a: int) -> int:
        r, c = _idx_to_grid(s, ncols)
        dr, dc = DIRS[a]
        rr, cc = _clip_move(r, c, dr, dc, nrows, ncols)
        return _grid_to_idx(rr, cc, ncols)

    for s in range(S):
        if absorbing[s]:
            # Absorbing states self-loop for all actions
            for a in ACTIONS:
                P[s, a, s] = 1.0
            continue

        for a in ACTIONS:
            if is_slippery:
                left = (a - 1) % 4
                right = (a + 1) % 4
                for aa in [left, a, right]:
                    s2 = step_from_state(s, aa)
                    P[s, a, s2] += 1.0/3.0
            else:
                s2 = step_from_state(s, a)
                P[s, a, s2] = 1.0

            # Reward for ARRIVING at goal
            for s2 in range(S):
                if P[s, a, s2] > 0:
                    rr, cc = _idx_to_grid(s2, ncols)
                    if grid[rr, cc] == 'G':
                        R[s, a, s2] = 1.0

    flatten_map = np.arange(S, dtype=int)
    return P, R, absorbing, (nrows, ncols), flatten_map

In [ ]:
P, R, absorbing, shape2d, flatmap = build_frozenlake_transitions(DESC_3x3, is_slippery=True)

In [ ]:
T_per_action = [P[:, a, :] for a in range(4)]  # list of 4 matrices, each (S,S)
P_all = np.array([T_per_action[0], T_per_action[1], T_per_action[2], T_per_action[3]])
print(P_all.shape)

In [ ]:
POLICY = [
    1,  # state 0 (top-left)
    2,  # state 1
    1,  # state 2
    1,  # state 3
    1,  # state 4 (center, possibly hole)
    1,  # state 5
    2,  # state 6
    2,  # state 7
    2,  # state 8 (goal state)
]

n_states = len(POLICY)
P_pi = np.zeros((n_states, n_states))
for s in range(n_states):
    a = POLICY[s]             # action chosen at state s
    P_pi[s, :] = P_all[a, s]  # copy the probabilities for that action

Reward = [
0,  # state 0 (top-left)
0,  # state 1
0,  # state 2
0,  # state 3
0,  # state 4 (center, possibly hole)
0,  # state 5
0,  # state 6
0,  # state 7
1,  # state 8 (goal state)
]
Reward = np.array(Reward)
gamma = 0.9

## 第一部分：迭代策略评估（Iterative Policy Evaluation）

在实验 2 中，我们使用闭式表达（closed-form expression）解析地求解了状态价值函数：

$$
V = (I - \gamma P_\pi)^{-1} R
$$

In [ ]:
# Solve for the value of the policy I designed 
I = np.eye(9)
A = I - gamma * P_pi
V = np.linalg.solve(A, Reward)
print("State values V:", V)

虽然这种方法是精确的，但它需要进行矩阵求逆，在状态空间较大时计算开销较高。  
一种替代方法是**迭代策略评估（iterative policy evaluation）**，即反复应用贝尔曼期望更新：

$$
V_{k+1}(s) \;=\; R(s) \;+\; \gamma \sum_{s'} P_\pi(s,s') \, V_k(s')
$$

### 步骤
1. 任意初始化价值函数（通常取 $V_0 = R$ 或 $V_0 = 0$）。  
2. 使用贝尔曼更新同时更新所有状态的价值。  
3. 重复多次迭代，直到价值函数收敛。  

在本练习中，我们将：
- 从 $V_0 = R$ 开始；  
- 运行约 50 次迭代；  
- 观察价值函数如何逐渐收敛到与实验 2 中闭式解相同的结果。  

In [ ]:
# Your time to work on it

## 第二部分：蒙特卡洛模拟（Monte Carlo Simulation）

在第一部分中，我们通过解析方法和迭代方法评估了一个固定策略。  
现在我们采用另一种思路：**仿真（simulation）**。

其核心思想是：在相同策略下，多次运行 FrozenLake 环境的完整回合（episodes），
通过对观测到的折扣回报进行平均，来估计每个状态的价值。

形式化地，状态价值定义为：

$$
V^\pi(s) = \mathbb{E}_\pi \Big[ \sum_{t=0}^{\infty} 
    \gamma^t R_{t+1} \;\Big|\; S_0 = s \Big]
$$

蒙特卡洛方法通过重复采样来近似这个期望：

1. 从给定状态 $s$ 开始；  
2. 按照策略 $\pi$ 运行一个完整回合，并记录奖励序列；  
3. 计算折扣回报  
   $G = r_1 + \gamma r_2 + \gamma^2 r_3 + \dots$；  
4. 重复多次，并取**平均回报**作为 $V^\pi(s)$ 的估计值。  

### 需要完成的内容
- 在给定策略下运行大量回合（例如 5000 次）；  
- 收集经验得到的状态价值；  
- 将结果与**第一部分**（迭代策略评估）以及**实验 2**（闭式解）进行比较。  

In [ ]:
# Your time to work on it
